In [1]:
import torch
import torchvision
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import pathlib

data_dir= pathlib.Path("../data")

In [2]:
train_data = datasets.Food101(root = data_dir,split="train",download=True,)
test_data = datasets.Food101(root=data_dir,split="test", download=True)

100%|██████████| 5.00G/5.00G [03:59<00:00, 20.9MB/s]


In [3]:
train_data

Dataset Food101
    Number of datapoints: 75750
    Root location: ../data
    split=train

In [4]:
class_names = train_data.classes
class_names

['apple_pie',
 'baby_back_ribs',
 'baklava',
 'beef_carpaccio',
 'beef_tartare',
 'beet_salad',
 'beignets',
 'bibimbap',
 'bread_pudding',
 'breakfast_burrito',
 'bruschetta',
 'caesar_salad',
 'cannoli',
 'caprese_salad',
 'carrot_cake',
 'ceviche',
 'cheese_plate',
 'cheesecake',
 'chicken_curry',
 'chicken_quesadilla',
 'chicken_wings',
 'chocolate_cake',
 'chocolate_mousse',
 'churros',
 'clam_chowder',
 'club_sandwich',
 'crab_cakes',
 'creme_brulee',
 'croque_madame',
 'cup_cakes',
 'deviled_eggs',
 'donuts',
 'dumplings',
 'edamame',
 'eggs_benedict',
 'escargots',
 'falafel',
 'filet_mignon',
 'fish_and_chips',
 'foie_gras',
 'french_fries',
 'french_onion_soup',
 'french_toast',
 'fried_calamari',
 'fried_rice',
 'frozen_yogurt',
 'garlic_bread',
 'gnocchi',
 'greek_salad',
 'grilled_cheese_sandwich',
 'grilled_salmon',
 'guacamole',
 'gyoza',
 'hamburger',
 'hot_and_sour_soup',
 'hot_dog',
 'huevos_rancheros',
 'hummus',
 'ice_cream',
 'lasagna',
 'lobster_bisque',
 'lobster

In [5]:
import random
import pathlib
amount_to_get = 1
data_path = data_dir / "food-101" / "images"
target_classes = ["fried_rice", "chicken_curry", "dumplings","donuts"]

# Change amount of data to get (e.g. 0.1 = random 10%, 0.2 = random 20%)
amount_to_get = 1
def get_subset(image_path=data_path,
               target_classes=["fried_rice", "chicken_curry", "dumplings", "donuts"],
               amount=0.1,
               val_split=0.2,
               seed=42):
    random.seed(seed)
    label_splits = {}

    # Process the standard files provided by Food-101
    for data_split in ["train", "test"]:
        label_path = data_dir / "food-101" / "meta" / f"{data_split}.txt"
        with open(label_path, "r") as f:
            labels = [line.strip("\n") for line in f.readlines() if line.split("/")[0] in target_classes]

        # Sample based on the 'amount' parameter
        number_to_sample = round(amount * len(labels))
        sampled_images = random.sample(labels, k=number_to_sample)

        # Logic to split Train into Train and Validation
        if data_split == "test":
            val_size = round(len(sampled_images) * val_split)
            # Shuffle is important before slicing
            random.shuffle(sampled_images)

            label_splits["val"] = sampled_images[:val_size]
            label_splits["test"] = sampled_images[val_size:]
        else:
            label_splits["train"] = sampled_images
    print(f"[INFO] Split train into: {len(label_splits['train'])} train, test into: {len(label_splits['test'])} train and {len(label_splits['val'])} validation images.")

    # Convert to full pathlib Paths
    full_path_splits = {}
    for split_name, images in label_splits.items():
        full_path_splits[split_name] = [pathlib.Path(str(image_path / img) + ".jpg") for img in images]

    return full_path_splits

label_splits = get_subset(amount=amount_to_get, val_split=0.5)
label_splits["train"][:10]

[INFO] Split train into: 3000 train, test into: 500 train and 500 validation images.


[PosixPath('../data/food-101/images/fried_rice/2704782.jpg'),
 PosixPath('../data/food-101/images/chicken_curry/3132208.jpg'),
 PosixPath('../data/food-101/images/chicken_curry/1498416.jpg'),
 PosixPath('../data/food-101/images/donuts/2661488.jpg'),
 PosixPath('../data/food-101/images/donuts/2160215.jpg'),
 PosixPath('../data/food-101/images/donuts/1800461.jpg'),
 PosixPath('../data/food-101/images/chicken_curry/3700392.jpg'),
 PosixPath('../data/food-101/images/chicken_curry/296213.jpg'),
 PosixPath('../data/food-101/images/fried_rice/3471388.jpg'),
 PosixPath('../data/food-101/images/dumplings/941945.jpg')]

In [6]:
# Create target directory path
target_dir_name = f"../data/fried_rice_chicken_curry_dumplings_donuts{str(int(amount_to_get*100))}_percent"
print(f"Creating directory: '{target_dir_name}'")

# Setup the directories
target_dir = pathlib.Path(target_dir_name)

# Make the directories
target_dir.mkdir(parents=True, exist_ok=True)

Creating directory: '../data/fried_rice_chicken_curry_dumplings_donuts100_percent'


In [7]:
import shutil

for image_split in label_splits.keys():
    for image_path in label_splits[str(image_split)]:
        dest_dir = target_dir / image_split / image_path.parent.stem / image_path.name
        if not dest_dir.parent.is_dir():
            dest_dir.parent.mkdir(parents=True, exist_ok=True)
        print(f"[INFO] Copying {image_path} to {dest_dir}...")
        shutil.copy2(image_path, dest_dir)



[INFO] Copying ../data/food-101/images/fried_rice/2704782.jpg to ../data/fried_rice_chicken_curry_dumplings_donuts100_percent/train/fried_rice/2704782.jpg...
[INFO] Copying ../data/food-101/images/chicken_curry/3132208.jpg to ../data/fried_rice_chicken_curry_dumplings_donuts100_percent/train/chicken_curry/3132208.jpg...
[INFO] Copying ../data/food-101/images/chicken_curry/1498416.jpg to ../data/fried_rice_chicken_curry_dumplings_donuts100_percent/train/chicken_curry/1498416.jpg...
[INFO] Copying ../data/food-101/images/donuts/2661488.jpg to ../data/fried_rice_chicken_curry_dumplings_donuts100_percent/train/donuts/2661488.jpg...
[INFO] Copying ../data/food-101/images/donuts/2160215.jpg to ../data/fried_rice_chicken_curry_dumplings_donuts100_percent/train/donuts/2160215.jpg...
[INFO] Copying ../data/food-101/images/donuts/1800461.jpg to ../data/fried_rice_chicken_curry_dumplings_donuts100_percent/train/donuts/1800461.jpg...
[INFO] Copying ../data/food-101/images/chicken_curry/3700392.jpg

In [8]:
# Zip pizza_steak_sushi images
zip_file_name = data_dir / f"../data/fried_rice_chicken_curry_dumplings_donuts{str(int(amount_to_get*100))}_percent"
shutil.make_archive(zip_file_name,
                    format="zip",
                    root_dir=target_dir)

'/data/fried_rice_chicken_curry_dumplings_donuts100_percent.zip'

In [9]:
from google.colab import drive
from google.colab import files
# drive.mount('/content/gdrive', force_remount=True)
files.download(zip_file_name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>